In [0]:
from pyspark.sql import functions as F

# Base paths for the lake layers
root   = "abfss://bikestores@oylamicdata.dfs.core.windows.net"
bronze = f"{root}/Bronze"
silver = f"{root}/Silver"

# Collects one row per step so we can save a pipeline log later
log = []

def log_step(table, step, rows_read, rows_rejected, rows_written):
    log.append((table, step, rows_read, rows_rejected, rows_written))
    print(f"{table:12} | {step:18} | read={rows_read} rejected={rows_rejected} written={rows_written}")

In [0]:
# ---------- CUSTOMERS ----------
df = spark.read.parquet(f"{bronze}/sales/customers")
n_read = df.count()

# 1. Remove duplicate customers (one row per customer_id)
df = df.dropDuplicates(["customer_id"])
n_dups = n_read - df.count()

# 2. Standardize data types and text formatting
df = (df
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("first_name", F.initcap(F.trim("first_name")))
    .withColumn("last_name",  F.initcap(F.trim("last_name")))
    .withColumn("email",      F.lower(F.trim("email")))
    .withColumn("street",     F.trim("street"))
    .withColumn("city",       F.trim("city"))
    .withColumn("state",      F.upper(F.trim("state")))
    .withColumn("zip_code",   F.trim("zip_code")))

# 3. Handle missing values
df = df.fillna({"phone": "unknown"})

# 4. Data quality check: id must exist and email must contain "@"
is_valid = F.col("customer_id").isNotNull() & F.col("email").contains("@")
valid   = df.filter(is_valid)
rejects = df.filter(~is_valid)

# 5. Write clean rows to Silver and bad rows to a rejects folder
valid.write.format("delta").mode("overwrite").save(f"{silver}/customers")
rejects.write.format("delta").mode("overwrite").save(f"{silver}/_rejects/customers")

# 6. Log the row counts
log_step("customers", "duplicates removed", n_read, n_dups, n_read - n_dups)
log_step("customers", "validate+write", n_read - n_dups, rejects.count(), valid.count())

customers    | duplicates removed | read=1445 rejected=0 written=1445
customers    | validate+write     | read=1445 rejected=0 written=1445


In [0]:
# ---------- ORDERS ----------
df = spark.read.parquet(f"{bronze}/sales/orders")
n_read = df.count()

# 1. Remove duplicate orders
df = df.dropDuplicates(["order_id"])
n_dups = n_read - df.count()

# 2. Standardize data types
df = (df
    .withColumn("order_id",      F.col("order_id").cast("int"))
    .withColumn("customer_id",   F.col("customer_id").cast("int"))
    .withColumn("order_status",  F.col("order_status").cast("int"))
    .withColumn("order_date",    F.to_date("order_date"))
    .withColumn("required_date", F.to_date("required_date"))
    .withColumn("shipped_date",  F.to_date("shipped_date")))

# 3. Missing values: shipped_date stays NULL for orders not shipped yet (valid business case)

# 4. Data quality checks
customers = spark.read.format("delta").load(f"{silver}/customers")

ok_basic = (
    F.col("order_id").isNotNull()
    & F.col("customer_id").isNotNull()
    & F.col("order_date").isNotNull()
    & (F.col("order_date") <= F.current_date())                       # no future order dates
    & (F.col("shipped_date").isNull() | (F.col("shipped_date") >= F.col("order_date")))  # ship after order
)

basic_valid = df.filter(ok_basic)
basic_bad   = df.filter(~ok_basic)

# Referential check: customer must exist in Silver customers
valid   = basic_valid.join(customers.select("customer_id"), "customer_id", "left_semi")
orphans = basic_valid.join(customers.select("customer_id"), "customer_id", "left_anti")
rejects = basic_bad.unionByName(orphans)

# 5. Write results
valid.write.format("delta").mode("overwrite").save(f"{silver}/orders")
rejects.write.format("delta").mode("overwrite").save(f"{silver}/_rejects/orders")

# 6. Log
log_step("orders", "duplicates removed", n_read, n_dups, n_read - n_dups)
log_step("orders", "validate+write", n_read - n_dups, rejects.count(), valid.count())

orders       | duplicates removed | read=1618 rejected=0 written=1618
orders       | validate+write     | read=1618 rejected=2 written=1616


In [0]:
# ---------- ORDER_ITEMS ----------
df = spark.read.parquet(f"{bronze}/sales/order_items")
n_read = df.count()

# 1. Remove duplicates (composite key: order_id + item_id)
df = df.dropDuplicates(["order_id", "item_id"])
n_dups = n_read - df.count()

# 2. Standardize data types
df = (df
    .withColumn("order_id",   F.col("order_id").cast("int"))
    .withColumn("item_id",    F.col("item_id").cast("int"))
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("quantity",   F.col("quantity").cast("int"))
    .withColumn("list_price", F.col("list_price").cast("decimal(10,2)"))
    .withColumn("discount",   F.col("discount").cast("decimal(4,2)")))

# 3. Missing values: treat missing discount as 0
df = df.fillna({"discount": 0})

# 4. Data quality checks
orders = spark.read.format("delta").load(f"{silver}/orders")

ok_basic = (
    F.col("order_id").isNotNull()
    & F.col("product_id").isNotNull()
    & (F.col("quantity") > 0)                                   # no zero/negative quantity
    & (F.col("list_price") >= 0)                                # no negative price
    & F.col("discount").between(0, 1)                           # discount must be 0 to 1
)

basic_valid = df.filter(ok_basic)
basic_bad   = df.filter(~ok_basic)

# Referential check: order must exist in Silver orders
valid   = basic_valid.join(orders.select("order_id"), "order_id", "left_semi")
orphans = basic_valid.join(orders.select("order_id"), "order_id", "left_anti")
rejects = basic_bad.unionByName(orphans)

# 5. Write results
valid.write.format("delta").mode("overwrite").save(f"{silver}/order_items")
rejects.write.format("delta").mode("overwrite").save(f"{silver}/_rejects/order_items")

# 6. Log
log_step("order_items", "duplicates removed", n_read, n_dups, n_read - n_dups)
log_step("order_items", "validate+write", n_read - n_dups, rejects.count(), valid.count())

order_items  | duplicates removed | read=4722 rejected=0 written=4722
order_items  | validate+write     | read=4722 rejected=2 written=4720


In [0]:
# ---------- DIMENSION TABLES: products, categories, brands, stores ----------
dims = {
    "products":   ("production/products",   ["product_id"]),
    "categories": ("production/categories", ["category_id"]),
    "brands":     ("production/brands",     ["brand_id"]),
    "stores":     ("sales/stores",          ["store_id"]),
}

for name, (path, key) in dims.items():
    df = spark.read.parquet(f"{bronze}/{path}")
    n_read = df.count()

    # 1. Remove duplicates on the business key
    df = df.dropDuplicates(key)

    # 2. Standardize text columns: trim spaces on every string column
    for c, t in df.dtypes:
        if t == "string":
            df = df.withColumn(c, F.trim(F.col(c)))

    # 3. Data quality check: key must not be null
    is_valid = F.lit(True)
    for k in key:
        is_valid = is_valid & F.col(k).isNotNull()

    # Extra rule for products: price must not be negative
    if name == "products":
        is_valid = is_valid & (F.col("list_price") >= 0)

    valid   = df.filter(is_valid)
    rejects = df.filter(~is_valid)

    # 4. Write results
    valid.write.format("delta").mode("overwrite").save(f"{silver}/{name}")
    rejects.write.format("delta").mode("overwrite").save(f"{silver}/_rejects/{name}")

    # 5. Log
    log_step(name, "clean+write", n_read, rejects.count(), valid.count())

products     | clean+write        | read=320 rejected=0 written=320
categories   | clean+write        | read=16 rejected=0 written=16
brands       | clean+write        | read=11 rejected=0 written=11
stores       | clean+write        | read=4 rejected=0 written=4


In [0]:
# Save the full Silver log as a table and show it
log_df = spark.createDataFrame(
    log, ["table", "step", "rows_read", "rows_rejected", "rows_written"]
)
log_df.write.format("delta").mode("overwrite").save(f"{silver}/_pipeline_log")
display(log_df)

table,step,rows_read,rows_rejected,rows_written
customers,duplicates removed,1445,0,1445
customers,validate+write,1445,0,1445
orders,duplicates removed,1618,0,1618
orders,validate+write,1618,2,1616
order_items,duplicates removed,4722,0,4722
order_items,validate+write,4722,2,4720
products,clean+write,320,0,320
categories,clean+write,16,0,16
brands,clean+write,11,0,11
stores,clean+write,4,0,4
